# LAB 03 — Retail Customer 360
## Data Cleansing, Quality, Deduplication & Slowly Changing Dimensions Type 2

**Environment:** Databricks Free Edition · Serverless · PySpark · Delta Lake · Unity Catalog  
**Source:** `retail_customer360.bronze.customers_raw` (from LAB 02)  
**Outputs:** `silver.customers_quarantine`, `silver.customers_superseded`, `silver.customers_scd2`, `silver.customers_current`  
**Language:** All explanations, comments, expected outcomes, and interview answers are in English.

### Business challenge
A retail company combines CRM, e-commerce, ERP, and marketing data. Customer records may be incomplete, inconsistent, duplicated, or updated over time. The platform must provide trusted current customer information **and** preserve changes that matter for historical analytics.

### Learning outcomes
- Transform PySpark DataFrames with built-in Spark expressions.
- Validate dates, emails, countries, and marketing consent.
- Quarantine invalid records with explainable error codes.
- Deduplicate valid candidates with a **deterministic** ranking rule.
- Build a Delta SCD Type 2 dimension using an **atomic staged MERGE**.
- Demonstrate current-versus-historical queries and **idempotency**.
- Explain when managed Lakeflow AUTO CDC would be a better production choice.

**Scope:** This lab uses a **full Bronze snapshot** to initialize SCD2, then a controlled two-record batch to demonstrate incremental behavior. The initial snapshot cannot reconstruct all changes that happened before the snapshot was taken.

**Safety:** The notebook does **not** drop or overwrite `silver.customers_scd2`. It overwrites only lab-owned **derived snapshots** (`customers_quarantine`, `customers_superseded`) and a dedicated **transient staging table**. The `customers_current` object is a view, not a duplicate Delta table.

## Architecture at a glance

```text
GitHub synthetic CRM CSV (LAB 02)
                |
        Bronze customers_raw
                |
          Normalize + cast
                |
       Data quality validation
                |
        +-------+--------+
        |                |
      VALID            INVALID
        |                |
    Deduplicate      Quarantine
        |                |
        +-----> Current candidates
                        |
                  SCD2 initial load
                        |
                Delta customers_scd2
                        ^
                        |
             Changed/new CRM events
                        |
               Staged Delta MERGE
                        |
            View silver.customers_current
                        |
                  Future Gold layer
```

**SCD2 invariant:** For a given business key, exactly one active row exists. Validity intervals use **`[effective_start, effective_end)`**; active rows have `effective_end = NULL`.

## Part I — Source inspection and environment

### Step 1 — Configure notebook context

Set fully qualified Unity Catalog identifiers. UTC gives a consistent interpretation of the CRM timestamps ending with `Z`. The target dimension is deliberately protected from full replacement.

In [0]:
# Step 1: Configure catalog, tables, and SQL session timezone.
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import DateType, TimestampType, BooleanType
from delta.tables import DeltaTable

CATALOG = "retail_customer360"
SOURCE_TABLE = f"{CATALOG}.bronze.customers_raw"
QUARANTINE_TABLE = f"{CATALOG}.silver.customers_quarantine"
SUPERSEDED_TABLE = f"{CATALOG}.silver.customers_superseded"
SCD2_TABLE = f"{CATALOG}.silver.customers_scd2"
CURRENT_VIEW = f"{CATALOG}.silver.customers_current"
STAGING_TABLE = f"{CATALOG}.silver._lab03_scd2_staging"
DEMO_BATCH_TABLE = f"{CATALOG}.silver._lab03_demo_batch"

spark.conf.set("spark.sql.session.timeZone", "UTC")
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql("USE SCHEMA silver")

# Confirm the notebook uses the intended governance context.
print("Current catalog:", spark.catalog.currentCatalog())
print("Current schema:", spark.catalog.currentDatabase())
print("Bronze input:", SOURCE_TABLE)
print("SCD2 history:", SCD2_TABLE)

Current catalog: retail_customer360
Current schema: silver
Bronze input: retail_customer360.bronze.customers_raw
SCD2 history: retail_customer360.silver.customers_scd2


### Step 2 — Read and inspect Bronze

Read the Delta table created in LAB 02. Do not download GitHub again: Bronze is the input contract for this lab. Require the original **15 CRM columns** plus the **5 LAB 02 ingestion metadata columns**.

In [0]:
# Step 2: Validate source table availability and required columns.
assert spark.catalog.tableExists(SOURCE_TABLE), (
    f"Missing {SOURCE_TABLE}. Complete LAB 02 first."
)

df_bronze = spark.table(SOURCE_TABLE)
CRM_COLUMNS = [
    "customer_id", "first_name", "last_name", "email", "phone",
    "date_of_birth", "country", "city", "postal_code", "customer_segment",
    "marketing_consent", "created_at", "updated_at", "source_system", "notes"
]
INGESTION_COLUMNS = [
    "ingestion_timestamp", "ingestion_batch_id", "source_file",
    "source_repository", "ingestion_layer"
]
missing = set(CRM_COLUMNS + INGESTION_COLUMNS) - set(df_bronze.columns)
assert not missing, f"LAB 02 contract mismatch; missing fields: {sorted(missing)}"

bronze_count = df_bronze.count()
assert bronze_count >= 1000, "This lab expects at least 1,000 incoming CRM rows."
print(f"Bronze rows: {bronze_count}; columns: {len(df_bronze.columns)}")
df_bronze.printSchema()
display(df_bronze.limit(12))

Bronze rows: 1223; columns: 20
root
 |-- customer_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- date_of_birth: string (nullable = true)
 |-- country: string (nullable = true)
 |-- city: string (nullable = true)
 |-- postal_code: string (nullable = true)
 |-- customer_segment: string (nullable = true)
 |-- marketing_consent: string (nullable = true)
 |-- created_at: string (nullable = true)
 |-- updated_at: string (nullable = true)
 |-- source_system: string (nullable = true)
 |-- notes: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- ingestion_batch_id: string (nullable = true)
 |-- source_file: string (nullable = true)
 |-- source_repository: string (nullable = true)
 |-- ingestion_layer: string (nullable = true)



customer_id,first_name,last_name,email,phone,date_of_birth,country,city,postal_code,customer_segment,marketing_consent,created_at,updated_at,source_system,notes,ingestion_timestamp,ingestion_batch_id,source_file,source_repository,ingestion_layer
null,Lucía,Apellido0001,customer0001@example.com,+34910000001,1970-01-12,ES,Madrid,28001,Silver,false,2024-01-02T09:00:00Z,2024-02-01T10:00:00Z,CRM_SIMULATED,Cliente sintético,2026-10-10T10:13:40.584Z,LAB02_BATCH_001,customers_lab02_v1.csv,github.com/normanmartinez2/retail_customer360,BRONZE
null,Mateo,Apellido0002,customer0002@example.com,+34910000002,1970-01-23,ES,Barcelona,08001,Gold,true,2024-01-03T09:00:00Z,2024-02-02T10:00:00Z,CRM_SIMULATED,Cliente sintético,2026-10-10T10:13:40.584Z,LAB02_BATCH_001,customers_lab02_v1.csv,github.com/normanmartinez2/retail_customer360,BRONZE
null,Zoë,Apellido0003,customer0003@example.com,+34910000003,1970-02-03,FR,Paris,75001,Standard,false,2024-01-04T09:00:00Z,2024-02-03T10:00:00Z,CRM_SIMULATED,Cliente sintético,2026-10-10T10:13:40.584Z,LAB02_BATCH_001,customers_lab02_v1.csv,github.com/normanmartinez2/retail_customer360,BRONZE
null,André,Apellido0004,customer0004@example.com,+34910000004,1970-02-14,PT,Lisboa,01000,Silver,true,2024-01-05T09:00:00Z,2024-02-04T10:00:00Z,CRM_SIMULATED,Cliente sintético,2026-10-10T10:13:40.584Z,LAB02_BATCH_001,customers_lab02_v1.csv,github.com/normanmartinez2/retail_customer360,BRONZE
null,Sofía,Apellido0005,customer0005@example.com,+34910000005,1970-02-25,DE,Berlin,10115,Gold,false,2024-01-06T09:00:00Z,2024-02-05T10:00:00Z,CRM_SIMULATED,Cliente sintético,2026-10-10T10:13:40.584Z,LAB02_BATCH_001,customers_lab02_v1.csv,github.com/normanmartinez2/retail_customer360,BRONZE
null,Łukasz,Apellido0006,customer0006@example.com,+34910000006,1970-03-08,ES,Madrid,28001,Standard,true,2024-01-07T09:00:00Z,2024-02-06T10:00:00Z,CRM_SIMULATED,Cliente sintético,2026-10-10T10:13:40.584Z,LAB02_BATCH_001,customers_lab02_v1.csv,github.com/normanmartinez2/retail_customer360,BRONZE
null,Marta,Apellido0007,customer0007@example.com,+34910000007,1970-03-19,ES,Barcelona,08001,Silver,false,2024-01-08T09:00:00Z,2024-02-07T10:00:00Z,CRM_SIMULATED,Cliente sintético,2026-10-10T10:13:40.584Z,LAB02_BATCH_001,customers_lab02_v1.csv,github.com/normanmartinez2/retail_customer360,BRONZE
null,Óscar,Apellido0008,customer0008@example.com,+34910000008,1970-03-30,FR,Paris,75001,Gold,true,2024-01-09T09:00:00Z,2024-02-08T10:00:00Z,CRM_SIMULATED,Cliente sintético,2026-10-10T10:13:40.584Z,LAB02_BATCH_001,customers_lab02_v1.csv,github.com/normanmartinez2/retail_customer360,BRONZE
null,Chloé,Apellido0009,customer0009@example.com,+34910000009,1970-04-10,PT,Lisboa,01000,Standard,false,2024-01-10T09:00:00Z,2024-02-09T10:00:00Z,CRM_SIMULATED,Cliente sintético,2026-10-10T10:13:40.584Z,LAB02_BATCH_001,customers_lab02_v1.csv,github.com/normanmartinez2/retail_customer360,BRONZE
null,Núria,Apellido0010,customer0010@example.com,+34910000010,1970-04-21,DE,Berlin,10115,Silver,true,2024-01-11T09:00:00Z,2024-02-10T10:00:00Z,CRM_SIMULATED,Cliente sintético,2026-10-10T10:13:40.584Z,LAB02_BATCH_001,customers_lab02_v1.csv,github.com/normanmartinez2/retail_customer360,BRONZE


## Part II — Cleansing and data-quality rules

### Step 3 — Normalize strings and represent blank values as NULL

`trim()` removes surrounding spaces; `lower()` standardizes email; `upper()` standardizes country. We preserve accented personal names and do **not** automatically title-case them. A blank email is allowed by our **lab** requirements; it does not mean the person can be emailed.

In [0]:
# Step 3: Standardize string fields without changing Bronze.
df_standardized = df_bronze
for field_name in CRM_COLUMNS:
    cleaned_value = F.trim(F.col(field_name))
    df_standardized = df_standardized.withColumn(
        field_name,
        F.when(cleaned_value == "", F.lit(None)).otherwise(cleaned_value)
    )

# Only standardize fields with well-defined casing rules.
df_standardized = (
    df_standardized
    .withColumn("email", F.lower(F.col("email")))
    .withColumn("country", F.upper(F.col("country")))
    .withColumn("marketing_consent", F.lower(F.col("marketing_consent")))
    .withColumn("customer_segment", F.initcap(F.lower(F.col("customer_segment"))))
)

display(df_standardized.select(
    "customer_id", "first_name", "email", "country", "marketing_consent"
).limit(20))

customer_id,first_name,email,country,marketing_consent
null,Lucía,customer0001@example.com,ES,false
null,Mateo,customer0002@example.com,ES,true
null,Zoë,customer0003@example.com,FR,false
null,André,customer0004@example.com,PT,true
null,Sofía,customer0005@example.com,DE,false
null,Łukasz,customer0006@example.com,ES,true
null,Marta,customer0007@example.com,ES,false
null,Óscar,customer0008@example.com,FR,true
null,Chloé,customer0009@example.com,PT,false
null,Núria,customer0010@example.com,DE,true


### Step 4 — Parse dates, timestamps, and consent safely

The incoming fields are strings, including values such as `1980-02-30`. `try_cast` returns NULL for malformed values rather than failing the entire query. Keep both raw and parsed fields until validation is complete. UTC parsing handles ISO timestamps such as `2024-01-02T09:00:00Z`.

In [0]:
# Step 4: Parse source types while preserving the input values.
df_typed = (
    df_standardized
    .withColumn("date_of_birth_parsed", F.expr("try_cast(date_of_birth AS DATE)"))
    .withColumn("created_at_parsed", F.expr("try_cast(created_at AS TIMESTAMP)"))
    .withColumn("updated_at_parsed", F.expr("try_cast(updated_at AS TIMESTAMP)"))
    .withColumn("marketing_consent_parsed", F.expr("try_cast(marketing_consent AS BOOLEAN)"))
)

display(df_typed.select(
    "customer_id", "date_of_birth", "date_of_birth_parsed",
    "updated_at", "updated_at_parsed", "marketing_consent", "marketing_consent_parsed"
).limit(20))

customer_id,date_of_birth,date_of_birth_parsed,updated_at,updated_at_parsed,marketing_consent,marketing_consent_parsed
null,1970-01-12,1970-01-12,2024-02-01T10:00:00Z,2024-02-01T10:00:00.000Z,false,false
null,1970-01-23,1970-01-23,2024-02-02T10:00:00Z,2024-02-02T10:00:00.000Z,true,true
null,1970-02-03,1970-02-03,2024-02-03T10:00:00Z,2024-02-03T10:00:00.000Z,false,false
null,1970-02-14,1970-02-14,2024-02-04T10:00:00Z,2024-02-04T10:00:00.000Z,true,true
null,1970-02-25,1970-02-25,2024-02-05T10:00:00Z,2024-02-05T10:00:00.000Z,false,false
null,1970-03-08,1970-03-08,2024-02-06T10:00:00Z,2024-02-06T10:00:00.000Z,true,true
null,1970-03-19,1970-03-19,2024-02-07T10:00:00Z,2024-02-07T10:00:00.000Z,false,false
null,1970-03-30,1970-03-30,2024-02-08T10:00:00Z,2024-02-08T10:00:00.000Z,true,true
null,1970-04-10,1970-04-10,2024-02-09T10:00:00Z,2024-02-09T10:00:00.000Z,false,false
null,1970-04-21,1970-04-21,2024-02-10T10:00:00Z,2024-02-10T10:00:00.000Z,true,true


### Step 5 — Declare and apply data-quality rules

These rules are explicit **lab business assumptions**, not universal Databricks standards.

| Code | Rule | Decision |
|---|---|---|
| DQ001 | Customer ID must be present | Quarantine |
| DQ002 | First name must be present | Quarantine |
| DQ003 | Last name must be present | Quarantine |
| DQ004 | Email must be well-formed **if present** | Quarantine |
| DQ005 | Informed birth date must parse | Quarantine |
| DQ006 | Birth date cannot be in the future | Quarantine |
| DQ007 | Country must be ES, PT, FR, or DE | Quarantine |
| DQ008 | Informed consent must be `true` or `false` | Quarantine |

A null email or birth date is permitted for this general Customer 360 dimension. Marketing eligibility requires separate rules.

In [0]:
# Step 5: Build one boolean flag per data-quality rule.
EMAIL_PATTERN = r"^[^@\s]+@[^@\s]+\.[^@\s]+$"
ALLOWED_COUNTRIES = ["ES", "PT", "FR", "DE"]

df_validated = (
    df_typed
    .withColumn("dq_missing_customer_id", F.col("customer_id").isNull())
    .withColumn("dq_missing_first_name", F.col("first_name").isNull())
    .withColumn("dq_missing_last_name", F.col("last_name").isNull())
    .withColumn(
        "dq_invalid_email",
        F.col("email").isNotNull() & ~F.regexp_like(F.col("email"), F.lit(EMAIL_PATTERN))
    )
    .withColumn(
        "dq_invalid_birth_date",
        F.col("date_of_birth").isNotNull() & F.col("date_of_birth_parsed").isNull()
    )
    .withColumn(
        "dq_future_birth_date",
        F.coalesce(F.col("date_of_birth_parsed") > F.current_date(), F.lit(False))
    )
    .withColumn(
        "dq_invalid_country",
        F.col("country").isNull() | ~F.col("country").isin(ALLOWED_COUNTRIES)
    )
    .withColumn(
        "dq_invalid_marketing_consent",
        F.col("marketing_consent").isNotNull() & ~F.col("marketing_consent").isin("true", "false")
    )
)

RULES = [
    ("DQ001", "dq_missing_customer_id"), ("DQ002", "dq_missing_first_name"),
    ("DQ003", "dq_missing_last_name"), ("DQ004", "dq_invalid_email"),
    ("DQ005", "dq_invalid_birth_date"), ("DQ006", "dq_future_birth_date"),
    ("DQ007", "dq_invalid_country"), ("DQ008", "dq_invalid_marketing_consent")
]

# Collect all violated codes; a row may fail more than one rule.
df_validated = (
    df_validated
    .withColumn(
        "dq_errors",
        F.array_compact(F.array(*[
            F.when(F.coalesce(F.col(flag), F.lit(False)), F.lit(code))
            for code, flag in RULES
        ]))
    )
    .withColumn(
        "dq_status",
        F.when(F.size("dq_errors") == 0, F.lit("VALID")).otherwise(F.lit("INVALID"))
    )
)

display(df_validated.select(
    "customer_id", "email", "date_of_birth", "country", "dq_errors", "dq_status"
).limit(25))

customer_id,email,date_of_birth,country,dq_errors,dq_status
null,customer0001@example.com,1970-01-12,ES,List(DQ001),INVALID
null,customer0002@example.com,1970-01-23,ES,List(DQ001),INVALID
null,customer0003@example.com,1970-02-03,FR,List(DQ001),INVALID
null,customer0004@example.com,1970-02-14,PT,List(DQ001),INVALID
null,customer0005@example.com,1970-02-25,DE,List(DQ001),INVALID
null,customer0006@example.com,1970-03-08,ES,List(DQ001),INVALID
null,customer0007@example.com,1970-03-19,ES,List(DQ001),INVALID
null,customer0008@example.com,1970-03-30,FR,List(DQ001),INVALID
null,customer0009@example.com,1970-04-10,PT,List(DQ001),INVALID
null,customer0010@example.com,1970-04-21,DE,List(DQ001),INVALID


### Step 6 — Profile rule violations

`explode()` turns error-code arrays into separate rows; a customer with two errors contributes two rule violations. Therefore, **the sum of rule violations is not necessarily the number of invalid rows**.

In [0]:
# Step 6: Profile valid/invalid rows and the error distribution.
display(df_validated.groupBy("dq_status").count().orderBy("dq_status"))

df_rule_counts = (
    df_validated.select(F.explode("dq_errors").alias("rule_code"))
    .groupBy("rule_code").count().orderBy("rule_code")
)
display(df_rule_counts)

dq_status,count
INVALID,50
VALID,1173


rule_code,count
DQ001,10
DQ002,3
DQ003,3
DQ004,12
DQ005,6
DQ006,4
DQ007,8
DQ008,10


### Step 7 — Split valid candidates from quarantine

Invalid records are **not deleted**; their original values and validation flags are available for investigation. Overwriting the lab quarantine table yields a **current batch snapshot**, not a historical remediation system.

In [0]:
# Step 7: Separate quality-compliant candidates and rejected rows.
df_valid_candidates = df_validated.filter(F.col("dq_status") == "VALID")
df_quarantine = df_validated.filter(F.col("dq_status") == "INVALID")

candidate_count = df_valid_candidates.count()
quarantine_count = df_quarantine.count()
assert bronze_count == candidate_count + quarantine_count

# Store all error codes and source metadata for root-cause analysis.
df_quarantine_output = df_quarantine.withColumn(
    "quarantine_timestamp", F.current_timestamp()
)
(df_quarantine_output.write.format("delta")
    .mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(QUARANTINE_TABLE))

print(f"Valid candidates: {candidate_count}; quarantined: {quarantine_count}")
display(spark.table(QUARANTINE_TABLE).select(
    "customer_id", "email", "country", "dq_errors"
).limit(20))

Valid candidates: 1173; quarantined: 50


customer_id,email,country,dq_errors
null,customer0001@example.com,ES,List(DQ001)
null,customer0002@example.com,ES,List(DQ001)
null,customer0003@example.com,FR,List(DQ001)
null,customer0004@example.com,PT,List(DQ001)
null,customer0005@example.com,DE,List(DQ001)
null,customer0006@example.com,ES,List(DQ001)
null,customer0007@example.com,ES,List(DQ001)
null,customer0008@example.com,FR,List(DQ001)
null,customer0009@example.com,PT,List(DQ001)
null,customer0010@example.com,DE,List(DQ001)


## Part III — Deduplication and initial Silver customer state

### Step 8 — Rank versions using a Window function

`dropDuplicates(['customer_id'])` does **not** guarantee that the most recent version is kept. Instead, order by parsed update time, creation time, and a deterministic fingerprint. Exact duplicate rows may still tie, but their tracked business values are identical.

**Important:** We use the **latest version per customer ID for the initial snapshot**. The initial load does not reconstruct every historical event previously emitted by the CRM.

In [0]:
# Step 8: Rank valid records by business key and recency.
ROW_FINGERPRINT_COLUMNS = CRM_COLUMNS

df_rank_input = df_valid_candidates.withColumn(
    "source_row_hash",
    F.sha2(F.to_json(
        F.struct(*[F.col(c) for c in ROW_FINGERPRINT_COLUMNS]),
        options={"ignoreNullFields": "false"}
    ), 256)
)

customer_window = Window.partitionBy("customer_id").orderBy(
    F.col("updated_at_parsed").desc_nulls_last(),
    F.col("created_at_parsed").desc_nulls_last(),
    F.col("source_row_hash").desc()
)

df_ranked = df_rank_input.withColumn(
    "row_num", F.row_number().over(customer_window)
)
display(df_ranked.select(
    "customer_id", "updated_at", "source_row_hash", "row_num"
).orderBy("customer_id", "row_num").limit(25))

customer_id,updated_at,source_row_hash,row_num
C000011,2024-02-11T10:00:00Z,5be1d45d9e0764fb738f11c19c8b59f800216952cda3d63dd796843c2a316c58,1
C000012,2024-02-12T10:00:00Z,8d0f0ffa26dcace3dde3f02b7a30684b34881fbe0b8f0bf0cc9f6c74c182db05,1
C000013,2024-02-13T10:00:00Z,7d34b4b951efb47071f46b7e26169f0ba28e4b9c7f076feacc1688ec66a9ad2a,1
C000014,2024-02-14T10:00:00Z,c103c496fd5c875811c4cb06116357ad4301641ee5d43de39bcb86e48db6ea72,1
C000015,2024-02-15T10:00:00Z,1ee7771af596a0443787be6781abd1a2519512bb1b3c612c9ce7a07c27ee0b4a,1
C000016,2024-02-16T10:00:00Z,880a5ef626faffe5de73abe513a7f06cee0d3f26788a551e7aa657ffaef8a075,1
C000017,2024-02-17T10:00:00Z,0aef049355a683359bc5b6457622da1ccab16901c9a26e5db827f053db8e1b17,1
C000018,2024-02-18T10:00:00Z,74be93ee12b4e1e957122de76c4632cc88e8c33b46c6796665bc043808a8a390,1
C000019,2024-02-19T10:00:00Z,cf42e9bcf6f153ca612d3f0c1d44535adc992cd7e2b51791072a7ccdfb61ede1,1
C000020,2024-02-20T10:00:00Z,38665f0d4fedc990574f7e64c83e04c820ab6665798672389ca2b30eb98fef7f,1


### Step 9 — Preserve superseded candidates

Rows ranked after the first are **valid, but not selected for the current representation**. Persisting them separately allows us to reconcile input rows and examine duplicates independently of quality failures.

In [0]:
# Step 9: Keep a single chosen row per customer ID; retain other versions.
df_deduplicated = df_ranked.filter(F.col("row_num") == 1)
df_superseded = df_ranked.filter(F.col("row_num") > 1)

initial_customer_count = df_deduplicated.count()
superseded_count = df_superseded.count()
assert candidate_count == initial_customer_count + superseded_count

(df_superseded.write.format("delta")
    .mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(SUPERSEDED_TABLE))

print("Initial unique customers:", initial_customer_count)
print("Superseded input rows:", superseded_count)
assert df_deduplicated.select("customer_id").distinct().count() == initial_customer_count

Initial unique customers: 1153
Superseded input rows: 20


### Step 10 — Build a typed, consumer-friendly customer snapshot

Keep original customer attributes and LAB 02 lineage metadata, but publish appropriately typed dates/timestamps and consent. This produces the **initial input** for our historical dimension; do not overwrite the SCD2 table with this DataFrame.

In [0]:
# Step 10: Select the business columns and correctly typed values.
CUSTOMER_COLUMNS = [
    "customer_id", "first_name", "last_name", "email", "phone",
    "date_of_birth", "country", "city", "postal_code", "customer_segment",
    "marketing_consent", "created_at", "updated_at", "source_system", "notes",
    "ingestion_timestamp", "ingestion_batch_id", "source_file", "source_repository",
    "silver_processed_timestamp"
]

df_customers_clean = df_deduplicated.select(
    "customer_id", "first_name", "last_name", "email", "phone",
    F.col("date_of_birth_parsed").alias("date_of_birth"),
    "country", "city", "postal_code", "customer_segment",
    F.col("marketing_consent_parsed").alias("marketing_consent"),
    F.col("created_at_parsed").alias("created_at"),
    F.col("updated_at_parsed").alias("updated_at"),
    "source_system", "notes", "ingestion_timestamp", "ingestion_batch_id",
    "source_file", "source_repository",
    F.current_timestamp().alias("silver_processed_timestamp")
)

df_customers_clean.printSchema()
assert isinstance(df_customers_clean.schema["date_of_birth"].dataType, DateType)
assert isinstance(df_customers_clean.schema["created_at"].dataType, TimestampType)
assert isinstance(df_customers_clean.schema["marketing_consent"].dataType, BooleanType)
display(df_customers_clean.limit(15))

root
 |-- customer_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- date_of_birth: date (nullable = true)
 |-- country: string (nullable = true)
 |-- city: string (nullable = true)
 |-- postal_code: string (nullable = true)
 |-- customer_segment: string (nullable = true)
 |-- marketing_consent: boolean (nullable = true)
 |-- created_at: timestamp (nullable = true)
 |-- updated_at: timestamp (nullable = true)
 |-- source_system: string (nullable = true)
 |-- notes: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- ingestion_batch_id: string (nullable = true)
 |-- source_file: string (nullable = true)
 |-- source_repository: string (nullable = true)
 |-- silver_processed_timestamp: timestamp (nullable = false)



customer_id,first_name,last_name,email,phone,date_of_birth,country,city,postal_code,customer_segment,marketing_consent,created_at,updated_at,source_system,notes,ingestion_timestamp,ingestion_batch_id,source_file,source_repository,silver_processed_timestamp
C000011,Lucía,Apellido0011,null,+34910000011,1970-05-02,ES,Madrid,28001,Gold,false,2024-01-12T09:00:00.000Z,2024-02-11T10:00:00.000Z,CRM_SIMULATED,Cliente sintético,2026-10-10T10:13:40.584Z,LAB02_BATCH_001,customers_lab02_v1.csv,github.com/normanmartinez2/retail_customer360,2026-10-10T11:56:53.448Z
C000012,Mateo,Apellido0012,null,+34910000012,1970-05-13,ES,Barcelona,08001,Standard,true,2024-01-13T09:00:00.000Z,2024-02-12T10:00:00.000Z,CRM_SIMULATED,Cliente sintético,2026-10-10T10:13:40.584Z,LAB02_BATCH_001,customers_lab02_v1.csv,github.com/normanmartinez2/retail_customer360,2026-10-10T11:56:53.448Z
C000013,Zoë,Apellido0013,null,+34910000013,1970-05-24,FR,Paris,75001,Silver,false,2024-01-14T09:00:00.000Z,2024-02-13T10:00:00.000Z,CRM_SIMULATED,Cliente sintético,2026-10-10T10:13:40.584Z,LAB02_BATCH_001,customers_lab02_v1.csv,github.com/normanmartinez2/retail_customer360,2026-10-10T11:56:53.448Z
C000014,André,Apellido0014,null,+34910000014,1970-06-04,PT,Lisboa,01000,Gold,true,2024-01-15T09:00:00.000Z,2024-02-14T10:00:00.000Z,CRM_SIMULATED,Cliente sintético,2026-10-10T10:13:40.584Z,LAB02_BATCH_001,customers_lab02_v1.csv,github.com/normanmartinez2/retail_customer360,2026-10-10T11:56:53.448Z
C000015,Sofía,Apellido0015,null,+34910000015,1970-06-15,DE,Berlin,10115,Standard,false,2024-01-16T09:00:00.000Z,2024-02-15T10:00:00.000Z,CRM_SIMULATED,Cliente sintético,2026-10-10T10:13:40.584Z,LAB02_BATCH_001,customers_lab02_v1.csv,github.com/normanmartinez2/retail_customer360,2026-10-10T11:56:53.448Z
C000016,Łukasz,Apellido0016,null,+34910000016,1970-06-26,ES,Madrid,28001,Silver,true,2024-01-17T09:00:00.000Z,2024-02-16T10:00:00.000Z,CRM_SIMULATED,Cliente sintético,2026-10-10T10:13:40.584Z,LAB02_BATCH_001,customers_lab02_v1.csv,github.com/normanmartinez2/retail_customer360,2026-10-10T11:56:53.448Z
C000017,Marta,Apellido0017,null,+34910000017,1970-07-07,ES,Barcelona,08001,Gold,false,2024-01-18T09:00:00.000Z,2024-02-17T10:00:00.000Z,CRM_SIMULATED,Cliente sintético,2026-10-10T10:13:40.584Z,LAB02_BATCH_001,customers_lab02_v1.csv,github.com/normanmartinez2/retail_customer360,2026-10-10T11:56:53.448Z
C000018,Óscar,Apellido0018,null,+34910000018,1970-07-18,FR,Paris,75001,Standard,true,2024-01-19T09:00:00.000Z,2024-02-18T10:00:00.000Z,CRM_SIMULATED,Cliente sintético,2026-10-10T10:13:40.584Z,LAB02_BATCH_001,customers_lab02_v1.csv,github.com/normanmartinez2/retail_customer360,2026-10-10T11:56:53.448Z
C000019,Chloé,Apellido0019,null,+34910000019,1970-07-29,PT,Lisboa,01000,Silver,false,2024-01-20T09:00:00.000Z,2024-02-19T10:00:00.000Z,CRM_SIMULATED,Cliente sintético,2026-10-10T10:13:40.584Z,LAB02_BATCH_001,customers_lab02_v1.csv,github.com/normanmartinez2/retail_customer360,2026-10-10T11:56:53.448Z
C000020,Núria,Apellido0020,null,+34910000020,1970-08-09,DE,Berlin,10115,Gold,true,2024-01-21T09:00:00.000Z,2024-02-20T10:00:00.000Z,CRM_SIMULATED,Cliente sintético,2026-10-10T10:13:40.584Z,LAB02_BATCH_001,customers_lab02_v1.csv,github.com/normanmartinez2/retail_customer360,2026-10-10T11:56:53.448Z


### Step 11 — Reconcile the initial Bronze snapshot

The initial full-snapshot balance must hold **before incremental SCD2 changes are applied**:

`bronze = quarantine + superseded + initial_unique_customers`.

Once updates are inserted into SCD2, do **not** compare the number of SCD2 history rows to the original Bronze row count.

In [0]:
# Step 11: Prove that every source row has a defined processing outcome.
assert bronze_count == quarantine_count + superseded_count + initial_customer_count, (
    "Unexpected record loss during validation/deduplication"
)
print("Bronze:", bronze_count)
print("Quarantine:", quarantine_count)
print("Superseded:", superseded_count)
print("Initial current customers:", initial_customer_count)
print("PASS: Initial source-to-Silver row reconciliation")

Bronze: 1223
Quarantine: 50
Superseded: 20
Initial current customers: 1153
PASS: Initial source-to-Silver row reconciliation


## Part IV — Slowly Changing Dimensions (SCD Type 2)

### Step 12 — Why SCD2 instead of a snapshot overwrite?

- **SCD Type 1:** replaces the former attribute values; there is no historical version in the dimension.
- **SCD Type 2:** closes the prior version and inserts a successor with a new validity interval.
- `whenMatchedUpdateAll()` + `whenNotMatchedInsertAll()` is generally an **SCD1-style upsert**, **not** a complete SCD2 implementation.

In our SCD2 dimension, each row has `effective_start`, `effective_end`, `is_current`, `version`, and `record_hash`. We use **half-open intervals**: `start <= event_time < end`.

**Limitation:** This simplified lab compares a single incoming version per customer to the **current** target version. It is not an out-of-order CDC engine. Late conflicting events are explicitly rejected below, not silently applied. For enterprise CDC, evaluate Lakeflow **AUTO CDC** with `STORED AS SCD TYPE 2`.

### Step 13 — Choose tracked attributes and compute a business hash

Only these **business** fields should trigger a new history version. Exclude technical ingestion timestamps and source extraction metadata; otherwise, retries would create false changes. The set of tracked attributes must be agreed with data owners.

In [0]:
# Step 13: Define tracked SCD2 columns and stable SHA-256 hashing.
TRACKED_COLUMNS = [
    "first_name", "last_name", "email", "phone", "date_of_birth",
    "country", "city", "postal_code", "customer_segment", "marketing_consent"
]

def add_business_hash(df):
    """Return a DataFrame with a stable fingerprint of tracked attributes."""
    return df.withColumn(
        "record_hash",
        F.sha2(F.to_json(
            F.struct(*[F.col(c) for c in TRACKED_COLUMNS]),
            options={"ignoreNullFields": "false"}
        ), 256)
    )

df_scd2_source = add_business_hash(df_customers_clean)

# Fix the output contract for each new SCD2 history row.
SCD2_COLUMNS = CUSTOMER_COLUMNS + [
    "record_hash", "effective_start", "effective_end", "is_current", "version"
]

display(df_scd2_source.select(
    "customer_id", "city", "customer_segment", "record_hash"
).limit(12))

customer_id,city,customer_segment,record_hash
C000011,Madrid,Gold,431355522a63ef7e8bcfad5bb70f17e57d0e66aad748462ef31fb479fe585d89
C000012,Barcelona,Standard,601061a2fc70a71f83047bcb14e7104fd7bfde196337227893c0a6f88b6ddc41
C000013,Paris,Silver,110c17c76914ce728b6f2b5ac0c40f680b008ec3847cfc25c6186e1f967e8c61
C000014,Lisboa,Gold,6816fc9431911c23adba635a598c1ea48e1f3f1ba9d3928a908142045d14ae35
C000015,Berlin,Standard,da3f490bd92d70338b9d23176dde45eeed7bf8be2541200fa349bd8c1365658f
C000016,Madrid,Silver,50abb8b2ac78b7970e7d3829c6b25efe7d1a2c35b30b4779caf29f7521733ce4
C000017,Barcelona,Gold,b4e7d789af942519b03e1705831ff94bc1da24bc363044c1210e59176b48239d
C000018,Paris,Standard,f3d948187e9b374fe335862453b53cb39c87e26368cbe5d196ae470c8f053a10
C000019,Lisboa,Silver,f224f533a6266e177ecc91a43781e2ad5d0d8e9503794896bd8face39d4c13aa
C000020,Berlin,Gold,b8af9504f11df6b0e79041920aca4ff36c167d950bf24ca4a63955fc9e7f7309


### Step 14 — Initialize the historical Delta dimension once

`effective_start` is the source update timestamp (fallback: creation timestamp, then ingestion timestamp). We create a **version 1** for each selected current customer. Existing history is **never overwritten**.

**Caution:** Time fallback to ingestion is a lab convenience; it is not a safe ordering substitute for a real CDC source sequence.

In [0]:
# Step 14: Build initial history only on the first run.
df_initial_scd2 = (
    df_scd2_source
    .withColumn(
        "effective_start",
        F.coalesce(F.col("updated_at"), F.col("created_at"), F.col("ingestion_timestamp"))
    )
    .withColumn("effective_end", F.lit(None).cast("timestamp"))
    .withColumn("is_current", F.lit(True))
    .withColumn("version", F.lit(1).cast("int"))
    .select(*SCD2_COLUMNS)
)

assert df_initial_scd2.filter(F.col("effective_start").isNull()).count() == 0

if not spark.catalog.tableExists(SCD2_TABLE):
    # Safe initial load: fail rather than replace if the table appears concurrently.
    (df_initial_scd2.write.format("delta")
        .mode("error").saveAsTable(SCD2_TABLE))
    print("Created initial SCD2 history:", SCD2_TABLE)
else:
    # This branch is expected when rerunning the notebook.
    required_scd2_columns = set(SCD2_COLUMNS)
    existing_columns = set(spark.table(SCD2_TABLE).columns)
    assert required_scd2_columns <= existing_columns, (
        "Existing SCD2 table has an incompatible schema; inspect it manually."
    )
    print("History already exists; initial load skipped (no overwrite).")

print("Total historical versions:", spark.table(SCD2_TABLE).count())
display(spark.table(SCD2_TABLE).select(
    "customer_id", "city", "effective_start", "effective_end", "is_current", "version"
).orderBy("customer_id").limit(12))

History already exists; initial load skipped (no overwrite).
Total historical versions: 1153


customer_id,city,effective_start,effective_end,is_current,version
C000011,Madrid,2024-02-11T10:00:00.000Z,null,true,1
C000012,Barcelona,2024-02-12T10:00:00.000Z,null,true,1
C000013,Paris,2024-02-13T10:00:00.000Z,null,true,1
C000014,Lisboa,2024-02-14T10:00:00.000Z,null,true,1
C000015,Berlin,2024-02-15T10:00:00.000Z,null,true,1
C000016,Madrid,2024-02-16T10:00:00.000Z,null,true,1
C000017,Barcelona,2024-02-17T10:00:00.000Z,null,true,1
C000018,Paris,2024-02-18T10:00:00.000Z,null,true,1
C000019,Lisboa,2024-02-19T10:00:00.000Z,null,true,1
C000020,Berlin,2024-02-20T10:00:00.000Z,null,true,1


## Part V — Controlled incremental update simulation

### Step 15 — Build two deterministic CRM examples

Create one **updated customer** and one **new customer** from a valid source record. This is a training batch, not actual Salesforce CDC.

The update changes `city` to `SCD2_DEMO_CITY`. The new customer uses the stable ID `LAB03_NEW_CUSTOMER`. On rerun, the business attributes remain the same, allowing an idempotency test.

We intentionally use fixed business values and a later event timestamp; do not generate random attributes or `current_timestamp()` as a CDC event time.

In [0]:
# Step 15: Select one existing customer and prepare two incoming events.
df_initial_current = spark.table(SCD2_TABLE).filter(F.col("is_current") == True)
sample_id = (
    df_initial_current
    .filter(F.col("customer_id") != "LAB03_NEW_CUSTOMER")
    .orderBy("customer_id")
    .select("customer_id").first()[0]
)
sample = df_initial_current.filter(F.col("customer_id") == sample_id)

# Case A: Same customer, new tracked attribute, later source update timestamp.
df_updated_customer = (
    sample
    .withColumn("city", F.lit("SCD2_DEMO_CITY"))
    .withColumn("updated_at", F.expr("effective_start + INTERVAL 1 DAY"))
    .select(*CUSTOMER_COLUMNS)
)

# Case B: New customer ID with a distinct current version.
df_new_customer = (
    sample
    .withColumn("customer_id", F.lit("LAB03_NEW_CUSTOMER"))
    .withColumn("first_name", F.lit("Demo"))
    .withColumn("last_name", F.lit("Customer"))
    .withColumn("email", F.lit("lab03.customer@example.com"))
    .withColumn("city", F.lit("Porto"))
    .withColumn("country", F.lit("PT"))
    .withColumn("updated_at", F.expr("effective_start + INTERVAL 1 DAY"))
    .select(*CUSTOMER_COLUMNS)
)

# Materialize the controlled input batch once before any history mutation.
# Reading this independent Delta table avoids a lazy read of the SCD2 target
# changing semantics between the first run and the idempotency replay.
df_demo_events_build = df_updated_customer.unionByName(df_new_customer)
(df_demo_events_build.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true").saveAsTable(DEMO_BATCH_TABLE))
df_demo_events = spark.table(DEMO_BATCH_TABLE)
assert df_demo_events.count() == 2
assert df_demo_events.select("customer_id").distinct().count() == 2
print("Existing customer selected:", sample_id)
display(df_demo_events.select("customer_id", "first_name", "city", "updated_at"))

Existing customer selected: C000011


customer_id,first_name,city,updated_at
LAB03_NEW_CUSTOMER,Demo,Porto,2024-02-12T10:00:00.000Z
C000011,Lucía,SCD2_DEMO_CITY,2024-02-12T10:00:00.000Z


### Step 16 — Implement the reusable, staged SCD2 MERGE

This is the most important coding exercise.

**Change detection:** `new key → insert`; `unchanged hash → ignore`; `changed hash with later source time → close current + insert successor`.

**Staged MERGE pattern:** Emit two source rows for each changed customer: one keyed row to **expire** the existing record, and one row with a NULL `merge_key` to **insert** the successor. New customers only get the NULL-key insertion row.

**Design safeguards:**
- Reject multiple incoming rows with the same business key in one micro-batch (deduplicate/sequence them upstream).
- Reject missing event times.
- **Raise an error** when a changed event arrives at or before the current effective start; this minimal example cannot correctly rewrite history out of order.
- Materialize source staging to a small dedicated Delta table **before** MERGE, avoiding a lazy self-read of the target during modification.
- `whenNotMatchedInsert` applies **only** to rows with `merge_key IS NULL`.
- The actual close+insert is one **atomic** Delta MERGE transaction.

**Lab limits:** One writer; a controlled batch; no hard deletes; source sequencing based on event timestamps; the staging table can be overwritten because it is *not* a history table.

In [0]:
# Step 16: Build a safe, replayable SCD2 merge function.
def process_scd2_batch(df_events, label="LAB03_DEMO"):
    """Apply a single deduplicated, ordered CRM batch to the SCD2 table.

    Reject conflicting late changes rather than silently corrupting historical
    validity intervals. Staging is materialized as Delta before target MERGE.
    """
    # Verify the minimum source contract and uniqueness for MERGE.
    missing_cols = set(CUSTOMER_COLUMNS) - set(df_events.columns)
    assert not missing_cols, f"Incremental source is missing: {sorted(missing_cols)}"
    assert df_events.filter(F.col("customer_id").isNull()).count() == 0
    assert (df_events.groupBy("customer_id").count()
        .filter(F.col("count") > 1).count()) == 0, (
            "The incremental batch must contain at most one event per customer ID."
        )

    # Track only meaningful business changes.
    incoming = (
        add_business_hash(df_events.select(*CUSTOMER_COLUMNS))
        .withColumn(
            "event_effective_start",
            F.coalesce(F.col("updated_at"), F.col("created_at"), F.col("ingestion_timestamp"))
        )
    )
    assert incoming.filter(F.col("event_effective_start").isNull()).count() == 0

    # Read the *current* SCD2 state for the business-key comparison.
    current = (
        spark.table(SCD2_TABLE)
        .filter(F.col("is_current") == True)
        .select(
            "customer_id",
            F.col("record_hash").alias("target_hash"),
            F.col("effective_start").alias("target_effective_start"),
            F.col("version").alias("target_version")
        )
    )

    compared = incoming.alias("src").join(
        current.alias("trg"), F.col("src.customer_id") == F.col("trg.customer_id"), "left"
    ).select("src.*", "trg.target_hash", "trg.target_effective_start", "trg.target_version")

    # A changed event older than the active version needs more advanced
    # out-of-order reconciliation. Fail fast instead of silently losing it.
    late_conflicts = compared.filter(
        F.col("target_hash").isNotNull()
        & (F.col("record_hash") != F.col("target_hash"))
        & (F.col("event_effective_start") <= F.col("target_effective_start"))
    )
    late_count = late_conflicts.count()
    if late_count:
        display(late_conflicts.select(
            "customer_id", "event_effective_start", "target_effective_start"
        ))
        raise ValueError(
            f"{late_count} conflicting late/equal-time event(s). "
            "Use explicit historical re-sequencing or Lakeflow AUTO CDC."
        )

    # A record is actionable only when it is new OR a later meaningful change.
    actionable = (
        compared.filter(
            F.col("target_hash").isNull()
            | (
                (F.col("record_hash") != F.col("target_hash"))
                & (F.col("event_effective_start") > F.col("target_effective_start"))
            )
        )
        .withColumn("effective_start", F.col("event_effective_start"))
        .withColumn("effective_end", F.lit(None).cast("timestamp"))
        .withColumn("is_current", F.lit(True))
        .withColumn("version", F.coalesce(F.col("target_version") + 1, F.lit(1)).cast("int"))
    )

    event_count = incoming.count()
    actionable_count = actionable.count()
    if actionable_count == 0:
        print(f"[{label}] {event_count} event(s); no meaningful new changes. MERGE skipped.")
        return {"events": event_count, "actionable": 0, "staged": 0, "merged": False}

    # Existing keys need an expiring row (match the current target version).
    to_expire = actionable.filter(F.col("target_hash").isNotNull()).select(
        F.col("customer_id").alias("merge_key"),
        *[F.col(name) for name in SCD2_COLUMNS]
    )

    # Every actionable event also needs a new current version, so use a null
    # merge key to force the NOT MATCHED insert branch.
    to_insert = actionable.select(
        F.lit(None).cast("string").alias("merge_key"),
        *[F.col(name) for name in SCD2_COLUMNS]
    )
    staged = to_expire.unionByName(to_insert)

    # Materialize the staged changes before modifying the source dimension.
    # This table is safe to replace: it is temporary LAB 03 staging only.
    (staged.write.format("delta").mode("overwrite")
        .option("overwriteSchema", "true").saveAsTable(STAGING_TABLE))
    staged_source = spark.table(STAGING_TABLE)
    staged_count = staged_source.count()
    print(f"[{label}] Events={event_count}; changes={actionable_count}; staged={staged_count}")
    display(staged_source.select(
        "merge_key", "customer_id", "city", "effective_start", "version"
    ))

    delta_target = DeltaTable.forName(spark, SCD2_TABLE)
    (
        delta_target.alias("trg")
        .merge(
            staged_source.alias("src"),
            "trg.customer_id = src.merge_key AND trg.is_current = true"
        )
        .whenMatchedUpdate(
            condition=(
                "trg.record_hash <> src.record_hash "
                "AND src.effective_start > trg.effective_start"
            ),
            set={
                "effective_end": "src.effective_start",
                "is_current": "false"
            }
        )
        .whenNotMatchedInsert(
            condition="src.merge_key IS NULL",
            values={name: f"src.{name}" for name in SCD2_COLUMNS}
        )
        .execute()
    )
    print(f"[{label}] SCD2 Delta MERGE committed.")
    return {"events": event_count, "actionable": actionable_count,
            "staged": staged_count, "merged": True}

### Step 17 — Execute the controlled MERGE

On a fresh SCD2 dimension, expect one new customer **and** one changed existing customer; the latter creates a second version. If you have already run this notebook, the same business changes should be detected as previously processed.

**Do not manually overwrite `customers_scd2` to reset it.** To start a different experiment, use a separate new catalog/table or design a deliberate reset strategy.

In [0]:
# Step 17: Apply the two simulated CRM events.
merge_result = process_scd2_batch(df_demo_events, "INITIAL_DEMO_BATCH")
print("MERGE outcome:", merge_result)

# Inspect the updated customer's complete version history.
display(
    spark.table(SCD2_TABLE)
    .filter(F.col("customer_id") == sample_id)
    .select("customer_id", "city", "effective_start", "effective_end", "is_current", "version")
    .orderBy("version")
)

[INITIAL_DEMO_BATCH] Events=2; changes=2; staged=3


merge_key,customer_id,city,effective_start,version
null,C000011,SCD2_DEMO_CITY,2024-02-12T10:00:00.000Z,2
null,LAB03_NEW_CUSTOMER,Porto,2024-02-12T10:00:00.000Z,1
C000011,C000011,SCD2_DEMO_CITY,2024-02-12T10:00:00.000Z,2


[INITIAL_DEMO_BATCH] SCD2 Delta MERGE committed.
MERGE outcome: {'events': 2, 'actionable': 2, 'staged': 3, 'merged': True}


customer_id,city,effective_start,effective_end,is_current,version
C000011,Madrid,2024-02-11T10:00:00.000Z,2024-02-12T10:00:00.000Z,false,1
C000011,SCD2_DEMO_CITY,2024-02-12T10:00:00.000Z,null,true,2


### Step 18 — Verify SCD2 history and newly inserted customer

The updated customer should retain the previous row as history, while the new row has `is_current = true`. A genuinely new business key starts at version 1.

In [0]:
# Step 18: Inspect old/current versions and the newly inserted business key.
display(
    spark.table(SCD2_TABLE)
    .filter(F.col("customer_id").isin(sample_id, "LAB03_NEW_CUSTOMER"))
    .select("customer_id", "first_name", "city", "effective_start",
            "effective_end", "is_current", "version")
    .orderBy("customer_id", "version")
)

assert spark.table(SCD2_TABLE).filter(
    (F.col("customer_id") == "LAB03_NEW_CUSTOMER") & F.col("is_current")
).count() == 1
assert spark.table(SCD2_TABLE).filter(
    (F.col("customer_id") == sample_id) & F.col("is_current")
).count() == 1
print("PASS: New customer and current-version uniqueness verified")

customer_id,first_name,city,effective_start,effective_end,is_current,version
C000011,Lucía,Madrid,2024-02-11T10:00:00.000Z,2024-02-12T10:00:00.000Z,false,1
C000011,Lucía,SCD2_DEMO_CITY,2024-02-12T10:00:00.000Z,null,true,2
LAB03_NEW_CUSTOMER,Demo,Porto,2024-02-12T10:00:00.000Z,null,true,1


PASS: New customer and current-version uniqueness verified


### Step 19 — Prove idempotency by replaying the same batch

Recompute change detection against the **new target state**; do **not** reuse a previously materialized `df_staged`. When business values have not changed, no new historical row must be inserted.

In [0]:
# Step 19: Re-run the same events and assert the history row count does not grow.
versions_before_replay = spark.table(SCD2_TABLE).count()
replay_result = process_scd2_batch(df_demo_events, "REPLAY_IDENTICAL_BATCH")
versions_after_replay = spark.table(SCD2_TABLE).count()

assert replay_result["actionable"] == 0, "The repeated batch was unexpectedly actionable."
assert versions_before_replay == versions_after_replay, "History grew on an identical replay."
print("PASS: Idempotent replay; historical version count unchanged")

[REPLAY_IDENTICAL_BATCH] 2 event(s); no meaningful new changes. MERGE skipped.
PASS: Idempotent replay; historical version count unchanged


### Step 20 — Demonstrate why we reject out-of-order conflicting events

A new value with an older timestamp cannot simply close the active version; it may need to be inserted **between** historical versions. The following test intentionally creates a late conflicting event and expects the function to raise an error **without writing** it. This guards historical correctness and illustrates why a managed CDC engine can be preferable.

In [0]:
# Step 20: Safely demonstrate late-event rejection without modifying SCD2.
current_sample = (
    spark.table(SCD2_TABLE)
    .filter((F.col("customer_id") == sample_id) & F.col("is_current"))
)
late_demo = (
    current_sample
    .withColumn("city", F.lit("CONFLICTING_LATE_CITY"))
    .withColumn("updated_at", F.expr("effective_start - INTERVAL 1 DAY"))
    .select(*CUSTOMER_COLUMNS)
)

before_late_test = spark.table(SCD2_TABLE).count()
try:
    process_scd2_batch(late_demo, "EXPECTED_LATE_EVENT_REJECTION")
    raise AssertionError("The late conflicting event was not rejected.")
except ValueError as exc:
    print("PASS: Late conflicting event rejected as designed:")
    print(str(exc))

assert spark.table(SCD2_TABLE).count() == before_late_test
print("PASS: No history was modified by the late-event test")

customer_id,event_effective_start,target_effective_start
C000011,2024-02-11T10:00:00.000Z,2024-02-12T10:00:00.000Z


PASS: Late conflicting event rejected as designed:
1 conflicting late/equal-time event(s). Use explicit historical re-sequencing or Lakeflow AUTO CDC.
PASS: No history was modified by the late-event test


## Part VI — Current view, historical queries, and integrity

### Step 21 — Publish current customers as a governed view

`customers_scd2` is the historical truth. Rather than duplicating active rows in a second physical table, publish a view that selects `is_current = true`.

If you previously created a physical table named `silver.customers_clean`, this notebook **does not drop it**. The current-view name is `silver.customers_current` to avoid collisions.

In [0]:
# Step 21: Create/refresh the customer-current view without touching history.
spark.sql(f"""
    CREATE OR REPLACE VIEW {CURRENT_VIEW} AS
    SELECT
        customer_id, first_name, last_name, email, phone, date_of_birth,
        country, city, postal_code, customer_segment, marketing_consent,
        created_at, updated_at, source_system, notes,
        ingestion_timestamp, ingestion_batch_id, source_file, source_repository,
        record_hash, effective_start, version
    FROM {SCD2_TABLE}
    WHERE is_current = true
""")

display(spark.table(CURRENT_VIEW).select(
    "customer_id", "first_name", "city", "country", "version"
).orderBy("customer_id").limit(25))

customer_id,first_name,city,country,version
C000011,Lucía,SCD2_DEMO_CITY,ES,2
C000012,Mateo,Barcelona,ES,1
C000013,Zoë,Paris,FR,1
C000014,André,Lisboa,PT,1
C000015,Sofía,Berlin,DE,1
C000016,Łukasz,Madrid,ES,1
C000017,Marta,Barcelona,ES,1
C000018,Óscar,Paris,FR,1
C000019,Chloé,Lisboa,PT,1
C000020,Núria,Berlin,DE,1


### Step 22 — Query customer state *as of* an event time

To join a fact with the correct historical dimension row, use the **half-open** validity predicate:

`fact.event_time >= dim.effective_start AND (fact.event_time < dim.effective_end OR dim.effective_end IS NULL)`.

Below, we query the selected customer's first version at its own effective start, then show the matching SCD2 history rows. A full as-of join with `orders_clean` is deferred until the multi-source labs.

In [0]:
# Step 22: Evaluate the dimension at the first known time for our sample key.
first_start = (
    spark.table(SCD2_TABLE)
    .filter(F.col("customer_id") == sample_id)
    .orderBy("effective_start", "version")
    .select("effective_start").first()[0]
)

as_of_rows = (
    spark.table(SCD2_TABLE)
    .filter(F.col("customer_id") == sample_id)
    .filter(F.col("effective_start") <= F.lit(first_start))
    .filter(F.col("effective_end").isNull() | (F.lit(first_start) < F.col("effective_end")))
)
print("As-of point (UTC):", first_start)
display(as_of_rows.select(
    "customer_id", "city", "effective_start", "effective_end", "version"
))
assert as_of_rows.count() == 1, "The as-of lookup must select one version."

As-of point (UTC): 2024-02-11 10:00:00


customer_id,city,effective_start,effective_end,version
C000011,Madrid,2024-02-11T10:00:00.000Z,2024-02-12T10:00:00.000Z,1


### Step 23 — Validate SCD2 invariants

Assertions cover:
1. One active row per business key.
2. Active rows have `effective_end = NULL`.
3. Expired rows have a non-null end greater than start.
4. No overlapping validity intervals.
5. Version sequence is contiguous (`1,2,3,...`) per customer.
6. The current view has the same row count as the active part of history.

These checks are useful, but do not replace full source-to-target CDC reconciliation and concurrent-writer controls.

In [0]:
# Step 23: Run executable SCD2 integrity checks.
history = spark.table(SCD2_TABLE)
active = history.filter(F.col("is_current") == True)
expired = history.filter(F.col("is_current") == False)

# Exactly one active row for each business key.
assert active.groupBy("customer_id").count().filter(F.col("count") > 1).count() == 0
assert active.select("customer_id").distinct().count() == history.select("customer_id").distinct().count(), (
    "Every customer in history must have a current version."
)
assert active.filter(F.col("effective_end").isNotNull()).count() == 0
assert active.filter(F.col("effective_start").isNull()).count() == 0
assert expired.filter(
    F.col("effective_end").isNull()
    | (F.col("effective_end") <= F.col("effective_start"))
).count() == 0

version_window = Window.partitionBy("customer_id").orderBy("effective_start", "version")
intervals = (
    history
    .withColumn("previous_start", F.lag("effective_start").over(version_window))
    .withColumn("previous_end", F.lag("effective_end").over(version_window))
    .withColumn("expected_version", F.row_number().over(version_window))
)

# Previous interval must close no later than the next interval opens.
overlap = intervals.filter(
    F.col("previous_start").isNotNull()
    & (F.col("previous_end").isNull()
       | (F.col("effective_start") < F.col("previous_end")))
)
assert overlap.count() == 0, "Overlapping SCD2 intervals detected"
assert intervals.filter(F.col("version") != F.col("expected_version")).count() == 0
assert spark.table(CURRENT_VIEW).count() == active.count()

print("PASS: SCD2 uniqueness, validity, versioning, and view checks")

PASS: SCD2 uniqueness, validity, versioning, and view checks


### Step 24 — Reconcile initial ingestion and historical outcomes

The **full Bronze snapshot** is reconciled against initial cleansing outcomes. The SCD2 table is a *different* metric: it includes additional historical versions created by the controlled incremental test.

**Do not assert** `bronze_rows = all_scd2_history_rows`; it would be conceptually incorrect.

In [0]:
# Step 24: Summarize the two distinct accounting views.
history_count = spark.table(SCD2_TABLE).count()
current_count = active.count()
expired_count = expired.count()

assert bronze_count == quarantine_count + superseded_count + initial_customer_count
assert history_count == current_count + expired_count

print("=" * 60)
print("RETAIL CUSTOMER 360 — LAB 03 VALIDATION")
print("=" * 60)
print("INITIAL SNAPSHOT ACCOUNTING")
print(f"Bronze: {bronze_count}")
print(f"Invalid/quarantined: {quarantine_count}")
print(f"Valid but superseded: {superseded_count}")
print(f"Initial distinct valid customers: {initial_customer_count}")
print("-" * 60)
print("HISTORICAL DIMENSION ACCOUNTING")
print(f"Total SCD2 versions: {history_count}")
print(f"Current versions: {current_count}")
print(f"Expired versions: {expired_count}")
print("=" * 60)
print("LAB 03 VALIDATION PASSED")

RETAIL CUSTOMER 360 — LAB 03 VALIDATION
INITIAL SNAPSHOT ACCOUNTING
Bronze: 1223
Invalid/quarantined: 50
Valid but superseded: 20
Initial distinct valid customers: 1153
------------------------------------------------------------
HISTORICAL DIMENSION ACCOUNTING
Total SCD2 versions: 1155
Current versions: 1154
Expired versions: 1
LAB 03 VALIDATION PASSED


### Step 25 — Inspect governance objects and transaction history

Unity Catalog provides the managed namespace for tables and views. Delta transaction history shows executed write operations. Read the history rather than assuming the MERGE happened.

In [0]:
# Step 25: Review Silver objects and the SCD2 transaction log.
display(spark.sql(f"SHOW TABLES IN {CATALOG}.silver"))
display(spark.sql(f"DESCRIBE DETAIL {SCD2_TABLE}"))
display(spark.sql(f"DESCRIBE HISTORY {SCD2_TABLE}"))

database,tableName,isTemporary
silver,_lab03_demo_batch,false
silver,_lab03_scd2_staging,false
silver,customers_current,false
silver,customers_quarantine,false
silver,customers_scd2,false
silver,customers_superseded,false


format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,355090b3-e819-4975-89fa-1472556ae60e,retail_customer360.silver.customers_scd2,null,,2026-10-10T11:55:12.119Z,2026-10-10T11:57:27.000Z,List(),List(),2,66296,"Map(delta.parquet.compression.codec -> zstd, delta.parquet.format.version.afe.internal -> 2.12.0, delta.enableDeletionVectors -> true, delta.parquet.format.version -> 2.12.0, io.unitycatalog.tableId -> 6a08fb21-97d6-4bfc-841c-7a889febbda2)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 1, numDeletionVectors -> 1)",false


version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
1,2026-10-10T11:57:27.000Z,72581255045954,norman.martinez2@gmail.com,MERGE,"Map(predicate -> [""((customer_id#25803 = merge_key#25854) AND is_current#25826)""], clusterBy -> [], matchedPredicates -> [{""predicate"":""(NOT (record_hash#25823 = record_hash#25875) AND (effective_start#25876 > effective_start#25824))"",""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""predicate"":""isnull(merge_key#25854)"",""actionType"":""insert""}])",null,List(4297527561571447),e383f716-c969-4480-aaf1-1e5f776c93d3,1010-111137-wjs8twpy-v2n,0,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 7858, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 1, executionTimeMs -> 5858, materializeSourceTimeMs -> 5, numTargetRowsInserted -> 2, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 2864, numTargetRowsUpdated -> 1, numOutputRows -> 3, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 3, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 2894)",null,Databricks-Runtime/19.9.x-aarch64-photon-scala2.13
0,2026-10-10T11:55:15.000Z,72581255045954,norman.martinez2@gmail.com,CREATE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""io.unitycatalog.tableId"":""6a08fb21-97d6-4bfc-841c-7a889febbda2""}, statsOnLoad -> true)",null,List(4297527561571447),b7cc8222-8421-47f0-8cc8-f0201d0e1e22,1010-111137-wjs8twpy-v2n,null,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1153, numOutputBytes -> 58438)",null,Databricks-Runtime/19.9.x-aarch64-photon-scala2.13


## Part VII — Interview readiness

### Step 26 — How to explain this solution as a Databricks Solutions Architect

**Q1. Why is `overwrite` not enough for SCD Type 2?**  
> `overwrite` can rebuild a current-state snapshot, but it does not by itself retain a business-validity history. SCD2 requires closing the old version and creating a new version when tracked attributes change.

**Q2. How does the staged Delta MERGE implement SCD2?**  
> A changed key generates one staged row to match and expire the current version, plus another with a null merge key to insert the successor. A new business key produces only the insert row. The close and insert execute atomically within Delta MERGE.

**Q3. Why use a record hash?**  
> It lets us compare a defined, stable set of business attributes. We exclude ingestion metadata to avoid false version creation during retries. The serialization and normalization rules must remain consistent.

**Q4. What if two events for a customer have the same timestamp or arrive out of order?**  
> We need reliable source sequencing, not an arbitrary tie-breaker. This simple current-row MERGE rejects conflicting late/equal-time updates. For genuine CDC streams, I would evaluate Lakeflow AUTO CDC with SCD Type 2 and source-provided sequencing.

**Q5. How do you prove idempotency?**  
> I replay the identical input batch after recomputing comparisons against the updated target and verify that no additional version is inserted.

**Q6. How would you model historical order analytics?**  
> Join each order to the customer dimension using the business key and the order timestamp inside the SCD2 validity interval. This preserves the customer attributes that were in effect when the order occurred.

**Q7. Would you always write this by hand?**  
> No. Manual Delta MERGE is helpful for understanding the mechanics and custom controlled pipelines. For operational CDC, Lakeflow AUTO CDC can reduce complexity and support SCD2 with out-of-order event handling.

### Official Databricks resources
- [Delta MERGE](https://docs.databricks.com/aws/en/delta/merge)
- [Lakeflow AUTO CDC](https://docs.databricks.com/aws/en/ldp/cdc)
- [Lakeflow CDC concepts](https://docs.databricks.com/aws/en/ldp/what-is-change-data-capture)
- [Unity Catalog](https://docs.databricks.com/aws/en/data-governance/unity-catalog/)
- [Medallion architecture](https://docs.databricks.com/aws/en/lakehouse/medallion)
- [PySpark DataFrame functions](https://docs.databricks.com/aws/en/pyspark/reference/functions/)

**Next:** LAB 04 should ingest versioned Products, Orders, Order Items, Payments, and Customer Interactions datasets from GitHub. Add real CDC event fields (`operation`, source event sequence, event time) before promoting SCD2 techniques to production.